# BRFSS Mental-Health EDA → Tableau Prep
**Goal:** profile the 1.1 GB BRFSS prevalence file *without* loading it all into RAM, verify the known traps, then export a small, clean, Tableau-ready CSV focused on mental health.

**Why DuckDB:** it runs SQL directly against the CSV on disk, so 1.1 GB never has to fit in memory. Pandas is used only for the small final slice.

**How to use:** run top to bottom. Read the printed output of Step 1 before running Step 3 (it tells you whether age-adjusted data exists, which you'll set in CONFIG).

In [2]:
# Install once if needed
# !pip install duckdb pandas
import duckdb, pandas as pd, numpy as np
print("duckdb", duckdb.__version__)

duckdb 1.4.0


## CONFIG — edit these

In [4]:
CSV_PATH          = "Behavioral_Risk_Factor_Surveillance_System_(BRFSS)_Prevalence_Data_(2011_to_present)_20260602.csv"   # <-- the 1.1 GB BRFSS file
TOPIC_KEYWORDS    = ["depress", "mental"]            # mental-health focus; broaden if you want
VALUE_TYPE        = "Crude Prevalence"               # AFTER Step 1, switch to "Age-adjusted Prevalence" if it exists
RELIABILITY_MIN_N = 50                                # CDC-style: estimates from n < 50 are unreliable

con = duckdb.connect()
SRC = f"read_csv_auto('{CSV_PATH}', sample_size=-1, all_varchar=true)"   # all_varchar avoids dtype crashes on a huge messy file

## Step 1 — Profile the file (memory-safe)
This resolves column names (the CSV download may use `Title_Case` while the API uses `lowercase`), then answers the questions that decide how you build:
- How big is it (years, states, topics)?
- Does **age-adjusted** prevalence exist, or only **crude**?
- What breakout categories exist (these are *separate slices* — never summed)?
- Is `Response` really just Yes/No complements?
- Which mental-health topics are available?
- How many rows are unreliable (small sample) or suppressed (footnoted)?

In [7]:
# resolve real column names regardless of casing
cols = con.execute(f"DESCRIBE SELECT * FROM {SRC}").df()
colnames = list(cols['column_name'])
def find(*cands):
    low = {c.lower(): c for c in colnames}
    for cand in cands:
        if cand.lower() in low: return low[cand.lower()]
    return None
C = {k: find(*v) for k, v in {
    'year':['Year','year'], 'state':['Locationdesc','locationdesc'], 'abbr':['Locationabbr','locationabbr'],
    'class':['Class','class'], 'topic':['Topic','topic'], 'question':['Question','question'],
    'response':['Response','response'], 'bo':['Break_Out','break_out'], 'boc':['Break_Out_Category','break_out_category'],
    'n':['Sample_Size','sample_size'], 'val':['Data_value','data_value'],
    'lo':['Confidence_limit_Low','confidence_limit_low'], 'hi':['Confidence_limit_High','confidence_limit_high'],
    'dvt':['Data_value_type','data_value_type'], 'foot':['Data_Value_Footnote_Symbol','data_value_footnote_symbol'],
}.items()}
print("Resolved columns:", C)

print("\n=== 1. SCALE ===")
print(con.execute(f"SELECT count(*) n_rows, count(distinct {C['year']}) n_years, "
                  f"count(distinct {C['state']}) n_states, count(distinct {C['topic']}) n_topics FROM {SRC}").df().to_string(index=False))
print("Year range:", con.execute(f"SELECT min({C['year']}), max({C['year']}) FROM {SRC}").fetchone())

print("\n=== 2. DATA_VALUE_TYPE (crude vs age-adjusted?) ===")
print(con.execute(f"SELECT {C['dvt']} AS dvt_type, count(*) n FROM {SRC} GROUP BY 1 ORDER BY 2 DESC").df().to_string(index=False))

print("\n=== 3. BREAKOUT CATEGORIES (each is a separate slice) ===")
print(con.execute(f"SELECT {C['boc']} AS category, count(distinct {C['bo']}) levels FROM {SRC} GROUP BY 1 ORDER BY 2 DESC").df().to_string(index=False))

print("\n=== 4. RESPONSE values ===")
print(con.execute(f"SELECT {C['response']} AS response_val, count(*) n FROM {SRC} GROUP BY 1 ORDER BY 2 DESC LIMIT 10").df().to_string(index=False))

print("\n=== 5. MENTAL-HEALTH topics available ===")
like = " OR ".join([f"lower({C['topic']}) LIKE '%{k}%'" for k in TOPIC_KEYWORDS])
print(con.execute(f"SELECT distinct {C['class']} AS class_name, {C['topic']} AS topic_name FROM {SRC} WHERE {like}").df().to_string(index=False))

print("\n=== 6. RELIABILITY: small-sample or suppressed rows ===")
print(con.execute(f"""SELECT count(*) total,
    sum(CASE WHEN TRY_CAST(replace({C['n']}, ',', '') AS INTEGER) < {RELIABILITY_MIN_N} THEN 1 ELSE 0 END) below_min_n,
    sum(CASE WHEN {C['val']} IS NULL OR {C['val']}='' THEN 1 ELSE 0 END) null_value,
    sum(CASE WHEN {C['foot']} IS NOT NULL AND {C['foot']}<>'' THEN 1 ELSE 0 END) footnoted
  FROM {SRC}""").df().to_string(index=False))

Resolved columns: {'year': 'Year', 'state': 'Locationdesc', 'abbr': 'Locationabbr', 'class': 'Class', 'topic': 'Topic', 'question': 'Question', 'response': 'Response', 'bo': 'Break_Out', 'boc': 'Break_Out_Category', 'n': 'Sample_Size', 'val': 'Data_value', 'lo': 'Confidence_limit_Low', 'hi': 'Confidence_limit_High', 'dvt': 'Data_value_type', 'foot': 'Data_Value_Footnote_Symbol'}

=== 1. SCALE ===
 n_rows  n_years  n_states  n_topics
2996473       14        56        66
Year range: ('2011', '2024')

=== 2. DATA_VALUE_TYPE (crude vs age-adjusted?) ===
        dvt_type       n
Crude Prevalence 2996473

=== 3. BREAKOUT CATEGORIES (each is a separate slice) ===
          category  levels
         Age Group      26
    Race/Ethnicity       8
  Household Income       8
Education Attained       4
               Sex       2
           Overall       1

=== 4. RESPONSE values ===
         response_val      n
                   No 660896
                  Yes 641343
              Married  19107
Go

## Step 2 — Decide before continuing
Look at the **Step 1 output #2**. If you see `Age-adjusted Prevalence`, go back to CONFIG and set `VALUE_TYPE = "Age-adjusted Prevalence"` — it's the correct choice for comparing states/groups because it removes the effect of one state simply being older than another. Re-run CONFIG. If only crude exists, leave it and note "crude" on the dashboard.

## Step 3 — Build the clean mental-health slice
Filters to your mental-health topics, **`Response = 'Yes'`** only (the `No` rows are just the complement), and one value type. Adds a `ci_width` and a `reliable` flag so unreliable estimates can be hidden or visibly marked in Tableau.

In [10]:
like = " OR ".join([f"lower({C['topic']}) LIKE '%{k}%'" for k in TOPIC_KEYWORDS])
df = con.execute(f"""
  SELECT {C['year']} AS Year, {C['abbr']} AS state_abbr, {C['state']} AS state,
         {C['class']} AS class, {C['topic']} AS topic, {C['question']} AS question,
         {C['bo']} AS breakout, {C['boc']} AS breakout_category, {C['dvt']} AS value_type,
         TRY_CAST(replace({C['n']}, ',', '') AS INTEGER) AS sample_n,
         TRY_CAST({C['val']} AS DOUBLE) AS value,
         TRY_CAST({C['lo']} AS DOUBLE) AS ci_low,
         TRY_CAST({C['hi']} AS DOUBLE) AS ci_high
  FROM {SRC}
  WHERE ({like}) AND {C['response']} = 'Yes' AND {C['dvt']} = '{VALUE_TYPE}'
""").df()
df["ci_width"] = df["ci_high"] - df["ci_low"]
df["reliable"] = (df["sample_n"] >= RELIABILITY_MIN_N) & (df["value"].notna())
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
print("slice rows:", len(df), "| reliable:", int(df['reliable'].sum()), "| dropped(null/small):", int((~df['reliable']).sum()))
df.head()

slice rows: 19104 | reliable: 15154 | dropped(null/small): 3950


,Year,state_abbr,state,class,topic,question,breakout,breakout_category,value_type,sample_n,value,ci_low,ci_high,ci_width,reliable
0,2024,AK,Alaska,Chronic Health Indicators,Depression,Ever told you that you have a form of depression?,Overall,Overall,Crude Prevalence,1109,21.9,20.2,23.6,3.4,True
1,2024,AK,Alaska,Chronic Health Indicators,Depression,Ever told you that you have a form of depression?,Male,Sex,Crude Prevalence,397,15.7,13.5,17.9,4.4,True
2,2024,AK,Alaska,Chronic Health Indicators,Depression,Ever told you that you have a form of depression?,Female,Sex,Crude Prevalence,712,28.9,26.2,31.6,5.4,True
3,2024,AK,Alaska,Chronic Health Indicators,Depression,Ever told you that you have a form of depression?,18-24,Age Group,Crude Prevalence,93,28.5,21.5,35.6,14.1,True
4,2024,AK,Alaska,Chronic Health Indicators,Depression,Ever told you that you have a form of depression?,25-34,Age Group,Crude Prevalence,180,28.7,23.8,33.5,9.7,True


## Step 4 — Insight seeds (for dashboard design)

In [12]:
latest = int(df["Year"].max())

print(f"--- State ranking, {latest}, Overall, reliable only ---")
r = df[(df.Year==latest) & (df.breakout=="Overall") & (df.reliable)].sort_values("value", ascending=False)
print(r[["state","topic","value","ci_low","ci_high","sample_n"]].head(15).to_string(index=False))

print("\n--- National trend over years (mean of state Overall values) ---")
print(df[(df.breakout=="Overall") & (df.reliable)].groupby(["topic","Year"])["value"].mean().round(1).to_string())

print("\n--- Disparity by Sex, latest year ---")
print(df[(df.Year==latest) & (df.breakout_category=="Sex") & (df.reliable)]
      .groupby(["topic","breakout"])["value"].mean().round(1).to_string())

print("\n--- Disparity by Race/Ethnicity, latest year (watch small n) ---")
print(df[(df.Year==latest) & (df.breakout_category=="Race/Ethnicity") & (df.reliable)]
      .groupby(["topic","breakout"])["value"].agg(["mean","count"]).round(1).to_string())

--- State ranking, 2024, Overall, reliable only ---
        state      topic  value  ci_low  ci_high  sample_n
West Virginia Depression   30.2    28.7     31.7      1570
     Kentucky Depression   28.3    26.8     29.8      2096
       Oregon Depression   27.3    25.9     28.6      1609
     Michigan Depression   26.7    25.5     27.9      2874
        Maine Depression   26.7    25.5     27.8      2988
    Louisiana Depression   26.6    25.0     28.2      1168
         Ohio Depression   26.6    25.4     27.9      2533
      Vermont Depression   26.4    25.0     27.9      1628
     Oklahoma Depression   26.1    24.9     27.4      1783
     Arkansas Depression   25.2    23.7     26.6      1278
 Rhode Island Depression   25.0    23.4     26.5      1370
      Alabama Depression   24.8    23.2     26.4      1187
      Montana Depression   24.7    23.3     26.0      1462
   Washington Depression   24.6    23.9     25.3      6116
      Indiana Depression   24.5    23.6     25.5      3010

---

## Step 5 — Export the Tableau-ready CSV (this is the file you publish from)

In [14]:
out = "brfss_mentalhealth_tableau.csv"
df.to_csv(out, index=False)
print("Exported", out, "| rows:", len(df), "| size MB:",
      round(__import__('os').path.getsize(out)/1e6, 2))
print("Columns:", list(df.columns))

Exported brfss_mentalhealth_tableau.csv | rows: 19104 | size MB: 3.56
Columns: ['Year', 'state_abbr', 'state', 'class', 'topic', 'question', 'breakout', 'breakout_category', 'value_type', 'sample_n', 'value', 'ci_low', 'ci_high', 'ci_width', 'reliable']


## Step 6 — Paste these back to me
Copy the printed output of: **Step 1 (all six sections)**, the **slice rows / reliable count** from Step 3, and the **three insight-seed tables** from Step 4. With those I'll tell you exactly what this data is, which mental-health story actually holds up statistically, and give you the four-dashboard Tableau spec with calculated-field formulas.

In [23]:
# ============================================================
# BRFSS Depression — Deeper EDA (DuckDB, memory-safe on 1.1 GB)
# Run once. Read G1, pick a correlate, set TOPIC_B/RESP_B, re-run G2.
# ============================================================
import duckdb, numpy as np

CSV_PATH          = "Behavioral_Risk_Factor_Surveillance_System_(BRFSS)_Prevalence_Data_(2011_to_present)_20260602.csv"   # <-- the 1.1 GB BRFSS file
TOPIC      = "Depression"
RESP       = "Yes"
VALUE_TYPE = "Crude Prevalence"
MIN_N      = 50
TOPIC_B, RESP_B = "Cost Barrier", "Yes"      # <-- set AFTER reading G1 (use a real topic name from the catalog)

con = duckdb.connect()
SRC = f"read_csv_auto('{CSV_PATH}', sample_size=-1, all_varchar=true)"
def base(extra=""):
    return (f"SELECT Year::INT yr, Locationabbr abbr, Locationdesc state, Break_Out bo, "
            f"Break_Out_Category boc, TRY_CAST(replace(Sample_Size,',','') AS INT) n, "
            f"TRY_CAST(Data_value AS DOUBLE) val FROM {SRC} "
            f"WHERE Topic='{TOPIC}' AND Response='{RESP}' AND Data_value_type='{VALUE_TYPE}' {extra}")

print("=== A. National 'US' row check + true trend vs state-mean ===")
print(con.execute(f"SELECT DISTINCT abbr, state FROM ({base()}) WHERE upper(state) LIKE '%UNITED STATES%' OR abbr IN('US','USA')").df().to_string(index=False))
print(con.execute(f"""WITH d AS ({base("AND Break_Out='Overall'")})
 SELECT yr, ROUND(MAX(CASE WHEN abbr IN('US','USA') THEN val END),1) us_row,
        ROUND(AVG(CASE WHEN abbr NOT IN('US','USA') THEN val END),1) state_mean
 FROM d GROUP BY yr ORDER BY yr""").df().to_string(index=False))

print("\n=== B. Income gradient (latest yr) ===")
print(con.execute(f"""WITH d AS ({base("AND boc='Household Income'")})
 SELECT bo income, ROUND(AVG(val),1) rate, SUM(n) tot_n FROM d
 WHERE yr=(SELECT MAX(yr) FROM d) AND n>={MIN_N} AND val IS NOT NULL
 GROUP BY bo ORDER BY rate DESC""").df().to_string(index=False))

print("\n=== C. Education gradient (latest yr) ===")
print(con.execute(f"""WITH d AS ({base("AND boc='Education Attained'")})
 SELECT bo education, ROUND(AVG(val),1) rate FROM d
 WHERE yr=(SELECT MAX(yr) FROM d) AND n>={MIN_N} AND val IS NOT NULL
 GROUP BY bo ORDER BY rate DESC""").df().to_string(index=False))

print("\n=== D. Sex gap over time (widening?) ===")
print(con.execute(f"""WITH d AS ({base("AND boc='Sex'")})
 SELECT yr, ROUND(AVG(CASE WHEN bo='Female' THEN val END),1) female,
        ROUND(AVG(CASE WHEN bo='Male' THEN val END),1) male,
        ROUND(AVG(CASE WHEN bo='Female' THEN val END)-AVG(CASE WHEN bo='Male' THEN val END),1) gap
 FROM d WHERE n>={MIN_N} GROUP BY yr ORDER BY yr""").df().to_string(index=False))

print("\n=== E. Fastest-rising states (first vs latest yr, Overall) ===")
print(con.execute(f"""WITH d AS ({base("AND Break_Out='Overall' AND abbr NOT IN('US','USA')")})
 SELECT state, ROUND(MAX(CASE WHEN yr=(SELECT MIN(yr) FROM d) THEN val END),1) first_yr,
        ROUND(MAX(CASE WHEN yr=(SELECT MAX(yr) FROM d) THEN val END),1) last_yr,
        ROUND(MAX(CASE WHEN yr=(SELECT MAX(yr) FROM d) THEN val END)
             -MAX(CASE WHEN yr=(SELECT MIN(yr) FROM d) THEN val END),1) delta
 FROM d GROUP BY state ORDER BY delta DESC NULLS LAST LIMIT 15""").df().to_string(index=False))

print("\n=== F. Age-group levels (watch for OVERLAPPING schemes — pick one clean set) ===")
print(con.execute(f"SELECT bo age_level, count(*) n FROM ({base("AND boc='Age Group'")}) GROUP BY 1 ORDER BY 1").df().to_string(index=False))

print("\n=== G1. Topic catalog (pick a correlate for G2) ===")
print(con.execute(f"SELECT Class AS class_name, Topic AS topic_name, count(distinct Locationdesc) states FROM {SRC} GROUP BY 1,2 ORDER BY 1,2").df().to_string(index=False))

print(f"\n=== G2. {TOPIC} vs {TOPIC_B} by state (latest yr, Overall) ===")
def overall(t,r):
    return (f"SELECT Locationabbr abbr, Year::INT yr, Locationdesc state, TRY_CAST(Data_value AS DOUBLE) val "
            f"FROM {SRC} WHERE Topic='{t}' AND Response='{r}' AND Data_value_type='{VALUE_TYPE}' "
            f"AND Break_Out='Overall' AND TRY_CAST(replace(Sample_Size,',','') AS INT)>={MIN_N}")
corr=con.execute(f"""WITH a AS ({overall(TOPIC,RESP)}), b AS ({overall(TOPIC_B,RESP_B)})
 SELECT a.state, ROUND(a.val,1) depression, ROUND(b.val,1) correlate
 FROM a JOIN b ON a.abbr=b.abbr AND a.yr=b.yr
 WHERE a.yr=(SELECT MAX(yr) FROM a) AND a.abbr NOT IN('US','USA') AND a.val IS NOT NULL AND b.val IS NOT NULL
 ORDER BY depression DESC""").df()
print(corr.to_string(index=False))
if len(corr)>=3:
    print(f"Pearson r = {round(np.corrcoef(corr['depression'],corr['correlate'])[0,1],3)}  (n states={len(corr)})")

print("\n=== H. Suppression by breakout category (for Dashboard 4) ===")
print(con.execute(f"""SELECT Break_Out_Category boc, count(*) total,
   ROUND(100.0*SUM(CASE WHEN Data_value IS NULL OR Data_value='' THEN 1 ELSE 0 END)/count(*),1) pct_suppressed,
   ROUND(100.0*SUM(CASE WHEN TRY_CAST(replace(Sample_Size,',','') AS INT)<{MIN_N} THEN 1 ELSE 0 END)/count(*),1) pct_small_n
 FROM {SRC} WHERE Topic='{TOPIC}' GROUP BY 1 ORDER BY pct_suppressed DESC""").df().to_string(index=False))

=== A. National 'US' row check + true trend vs state-mean ===
abbr                                      state
  US All States, DC and Territories (median) **
  yr  us_row  state_mean
2011    17.5        17.7
2012    17.6        17.9
2013    18.7        18.8
2014    18.7        18.8
2015    18.9        18.8
2016    17.3        17.7
2017    20.0        20.0
2018    19.6        19.2
2019    19.7        19.8
2020    19.2        19.2
2021    20.4        20.3
2022    21.6        21.2
2023    21.8        21.2
2024    21.7        21.8

=== B. Income gradient (latest yr) ===
           income  rate   tot_n
Less than $15,000  37.0  7415.0
  $15,000-$24,999  30.6  9870.0
  $25,000-$34,999  26.9 10353.0
  $35,000-$49,999  24.4 11077.0
  $50,000-$99,999  21.9 22729.0
$100,000-$199,999  18.9 14761.0
        $200,000+  15.3  3630.0

=== C. Education gradient (latest yr) ===
       education  rate
  Some post-H.S.  24.5
  Less than H.S.  23.6
  H.S. or G.E.D.  21.6
College graduate  19.8

=== D. Sex g

In [25]:
# --- Step 1: find the right response value for your correlate candidates ---
for t in ["Health Care Cost", "Disability status", "Fair or Poor Health"]:
    vals = con.execute(f"SELECT DISTINCT Response FROM {SRC} WHERE Topic='{t}'").df()['Response'].tolist()
    print(t, "->", vals)

# --- Step 2: set TOPIC_B / RESP_B from what prints above, then re-run the G2 block ---
# e.g.  TOPIC_B, RESP_B = "Health Care Cost", "Yes"

# --- Bonus: the age pattern you haven't pulled yet (clean 18-24...65+ set) ---
age = con.execute(f"""
 WITH d AS ({base("AND boc='Age Group'")})
 SELECT bo age_group, ROUND(AVG(val),1) rate, SUM(n) tot_n FROM d
 WHERE yr=(SELECT MAX(yr) FROM d) AND n>={MIN_N} AND val IS NOT NULL
 GROUP BY bo ORDER BY age_group""").df()
print(age.to_string(index=False))

Health Care Cost -> ['No', 'Yes']
Disability status -> ['No', 'Yes']
Fair or Poor Health -> ['Fair or Poor Health', 'Good or Better Health']
age_group  rate   tot_n
    18-24  27.4  7131.0
    25-34  27.4 13481.0
    35-44  24.7 15346.0
    45-54  22.3 15157.0
    55-64  20.9 17806.0
      65+  15.3 26701.0


In [27]:
import numpy as np
def overall(t, r):
    return (f"SELECT Locationabbr abbr, Year::INT yr, Locationdesc state, TRY_CAST(Data_value AS DOUBLE) val "
            f"FROM {SRC} WHERE Topic='{t}' AND Response='{r}' AND Data_value_type='{VALUE_TYPE}' "
            f"AND Break_Out='Overall' AND TRY_CAST(replace(Sample_Size,',','') AS INT)>={MIN_N}")

candidates = [("Health Care Cost", "Yes"),
              ("Disability status", "Yes"),
              ("Fair or Poor Health", "Fair or Poor Health")]

results = {}
for tb, rb in candidates:
    df = con.execute(f"""WITH a AS ({overall(TOPIC, RESP)}), b AS ({overall(tb, rb)})
     SELECT a.state, ROUND(a.val,1) depression, ROUND(b.val,1) correlate
     FROM a JOIN b ON a.abbr=b.abbr AND a.yr=b.yr
     WHERE a.yr=(SELECT MAX(yr) FROM a) AND a.abbr NOT IN('US','USA')
       AND a.val IS NOT NULL AND b.val IS NOT NULL ORDER BY depression DESC""").df()
    if len(df) >= 3:
        r = np.corrcoef(df['depression'], df['correlate'])[0, 1]
        results[tb] = (round(r, 3), len(df), df)
        print(f"{tb:22s} Pearson r = {r:+.3f}  (n states = {len(df)})")

# full scatter table for the strongest |r|
best = max(results, key=lambda k: abs(results[k][0]))
print(f"\n--- Strongest: {best} (r={results[best][0]}) — state-by-state ---")
print(results[best][2].to_string(index=False))

Health Care Cost       Pearson r = -0.061  (n states = 54)
Disability status      Pearson r = +0.067  (n states = 266)
Fair or Poor Health    Pearson r = +0.124  (n states = 54)

--- Strongest: Fair or Poor Health (r=0.124) — state-by-state ---
                        state  depression  correlate
                West Virginia        30.2       26.3
                     Kentucky        28.3       23.9
                       Oregon        27.3       20.7
                        Maine        26.7       19.2
                     Michigan        26.7       21.1
                         Ohio        26.6       19.1
                    Louisiana        26.6       24.1
                      Vermont        26.4       13.5
                     Oklahoma        26.1       20.9
                     Arkansas        25.2       24.7
                 Rhode Island        25.0       17.3
                      Alabama        24.8       22.1
                      Montana        24.7       18.0
             